In [2]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time, os

uploaded = files.upload()  # select all six CSVs
spark = SparkSession.builder.appName("EnergyAssessment").getOrCreate()
def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)

Saving substation_master.csv to substation_master.csv
Saving tariff_master.csv to tariff_master.csv


**SECTION 01 / DATA PREPARATION & ANALYSIS**

**Tasks 01-13**

**A. Inspect, select and transform**

01. Load all six CSVs. Display the schema, row count and first 10 rows of the main readings DataFrame.

In [3]:
readings_df    = load_csv("meter_readings_90000.csv")
readings_extra_df       = load_csv("meter_readings_extra_12000.csv")
meter_master   = load_csv("meter_master.csv")
substation_df  = load_csv("substation_master.csv")
tariff_df      = load_csv("tariff_master.csv")
tickets_df     = load_csv("maintenance_tickets_1600.csv")

readings_df.printSchema()
print("Row count:", readings_df.count())
readings_df.show(10)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Row count: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-04

02. Select reading_id, meter_id, reading_timestamp, energy_kwh and reading_status. In a separate
DataFrame, rename source_type to collection_mode.

In [7]:
selected_df = readings_df.select("reading_id", "meter_id", "reading_timestamp", "energy_kwh", "reading_status")
selected_df.show(5)
renamed_df = readings_df.withColumnRenamed("source_type", "collection_mode")
print(renamed_df.printSchema())
renamed_df.show(5)

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|         Valid|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|         Valid|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|     Estimated|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|         Valid|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|         Valid|
+----------+--------+-------------------+----------+--------------+
only showing top 5 rows
root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- 

03. Filter readings where reading_status is Valid and energy_kwh exceeds 20. Sort by energy consumed,
highest first.

In [8]:
high_valid_df = readings_df.filter((F.col("reading_status") == "Valid") & (F.col("energy_kwh") > 20)).orderBy(F.col("energy_kwh").desc())
high_valid_df.show(10)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid| SmartMeter|                 31.1|      OP010|
| RD0077494|   M0014|2026-05-05 17:30:00|    109.99|     NULL|       0.771|             0|         Valid|     Manual|                 39.0|      OP031|
| RD0039063|   M0101|2026-02-01 00:45:00|    109.96|    231.6|       0.835|             0|         Valid| SmartMeter|                 20.1|      OP021|
| RD0034412|   M0026|2026-02-24 21:15:00|    109.96|    231.6|       0.964|             

04. Create energy_band using when() / otherwise(): Low below 5 kWh, Medium from 5 through 25 kWh,
High above 25 kWh, and Unknown for nulls.

In [9]:
banded_df = readings_df.withColumn("energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium")
     .otherwise("High"))
banded_df.groupBy("energy_band").count().show()

+-----------+-----+
|energy_band|count|
+-----------+-----+
|       High|21751|
|        Low|26898|
|    Unknown| 1620|
|     Medium|39731|
+-----------+-----+



**B. Null values and data cleaning**

05. Calculate null counts for energy_kwh, voltage_v, power_factor and operator_id.

In [10]:
null_cols = ["energy_kwh", "voltage_v", "power_factor", "operator_id"]
readings_df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls") for c in null_cols]).show()

+----------------+---------------+------------------+-----------------+
|energy_kwh_nulls|voltage_v_nulls|power_factor_nulls|operator_id_nulls|
+----------------+---------------+------------------+-----------------+
|            1620|           2070|              1440|             5400|
+----------------+---------------+------------------+-----------------+



06. Create a new DataFrame that fills missing operator_id with UNASSIGNED and missing power_factor with
0.90. Do not modify the original DataFrame.

In [11]:
filled_df = readings_df.fillna({"operator_id": "UNASSIGNED", "power_factor": 0.90})
filled_df.select(*null_cols).agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls") for c in null_cols]).show()
# original unchanged:
print("Original operator_id nulls:", readings_df.filter(F.col("operator_id").isNull()).count())

+----------------+---------------+------------------+-----------------+
|energy_kwh_nulls|voltage_v_nulls|power_factor_nulls|operator_id_nulls|
+----------------+---------------+------------------+-----------------+
|            1620|           2070|                 0|                0|
+----------------+---------------+------------------+-----------------+

Original operator_id nulls: 5400


07. Remove readings with missing energy_kwh or voltage_v. Report the number of rows removed.

In [12]:
before = readings_df.count()
clean_df = readings_df.dropna(subset=["energy_kwh", "voltage_v"])
after = clean_df.count()
print(f"Before: {before} | After: {after} | Removed: {before - after}")

Before: 90000 | After: 86349 | Removed: 3651


**C. Joins, groupBy and aggregations**

In [13]:
# Slim dimension tables so joins never create duplicate column names
meter_dim  = meter_master.select("meter_id", "site_type", "substation_id", "tariff_id")
sub_dim    = substation_df.select("substation_id", "city", "region")
tariff_dim = tariff_df.select("tariff_id", "rate_per_kwh_inr")

08. Join readings with meter_master on meter_id. Calculate total energy for Valid readings by site_type.

In [14]:
readings_meter_master_df = readings_df.join(meter_dim, on="meter_id", how="inner")
valid_rm = readings_meter_master_df.filter(F.col("reading_status") == "Valid")
(valid_rm.groupBy("site_type")
    .agg(F.round(F.sum("energy_kwh"), 2).alias("total_valid_kwh"))
    .orderBy(F.desc("total_valid_kwh")).show())

+---------------+---------------+
|      site_type|total_valid_kwh|
+---------------+---------------+
|     Industrial|      710317.41|
|     Commercial|      451525.18|
|Public Facility|      138870.41|
|    Residential|      100863.29|
+---------------+---------------+



09. Extend the join to substation_master. Calculate total Valid energy by region and city.

In [16]:
rms_df = readings_meter_master_df.join(sub_dim, on="substation_id", how="inner")
(rms_df.filter(F.col("reading_status") == "Valid")
    .groupBy("region", "city")
    .agg(F.round(F.sum("energy_kwh"), 2).alias("total_valid_kwh"))
    .orderBy("region", F.desc("total_valid_kwh")).show())

+-------+-------------+---------------+
| region|         city|total_valid_kwh|
+-------+-------------+---------------+
|Central|       Indore|      176503.91|
|  North|      Lucknow|       142610.4|
|  North|        Delhi|       94234.89|
|  North|       Jaipur|       47863.16|
|  South|Visakhapatnam|      205015.83|
|  South|        Kochi|      204272.46|
|  South|    Bengaluru|        73787.6|
|  South|    Hyderabad|       51937.79|
|  South|      Chennai|       51829.06|
|   West|         Pune|      162777.24|
|   West|    Ahmedabad|      138627.74|
|   West|       Mumbai|       52116.21|
+-------+-------------+---------------+



10. Join tariff_master through meter_master. For non-Rejected readings with non-null energy, calculate
energy_charge_inr = energy_kwh * rate_per_kwh_inr. Do not add the monthly fixed charge to every
reading.

In [17]:
rmt_df = readings_meter_master_df.join(tariff_dim, on="tariff_id", how="inner")
billable_df = (rmt_df
    .filter((F.col("reading_status") != "Rejected") & F.col("energy_kwh").isNotNull())
    .withColumn("energy_charge_inr", F.col("energy_kwh") * F.col("rate_per_kwh_inr")))
billable_df.select("reading_id", "meter_id", "energy_kwh", "rate_per_kwh_inr", "energy_charge_inr").show(10)
print("Total energy charge (INR):", billable_df.agg(F.round(F.sum("energy_charge_inr"), 2)).first()[0])

+----------+--------+----------+----------------+------------------+
|reading_id|meter_id|energy_kwh|rate_per_kwh_inr| energy_charge_inr|
+----------+--------+----------+----------------+------------------+
| RD0000001|   M0082|     22.53|             6.7|150.95100000000002|
| RD0000002|   M0020|      2.97|             6.4|19.008000000000003|
| RD0000003|   M0054|      8.61|             8.2| 70.60199999999999|
| RD0000004|   M0111|      1.87|             5.1| 9.536999999999999|
| RD0000005|   M0135|      4.39|             8.2|            35.998|
| RD0000006|   M0139|      0.36|             6.4|             2.304|
| RD0000007|   M0142|     79.47|             9.1| 723.1769999999999|
| RD0000009|   M0005|      1.96|             5.1| 9.995999999999999|
| RD0000010|   M0052|     22.52|             6.7|150.88400000000001|
| RD0000011|   M0137|     10.51|             6.7|            70.417|
+----------+--------+----------+----------------+------------------+
only showing top 10 rows
Total ene

11. Join maintenance tickets with meter_master. Find ticket counts and average maintenance_cost_inr by
site_type and severity.

In [18]:
tm_df = tickets_df.join(meter_dim.select("meter_id", "site_type"), on="meter_id", how="inner")
(tm_df.groupBy("site_type", "severity")
    .agg(F.count("*").alias("ticket_count"),
         F.round(F.avg("maintenance_cost_inr"), 2).alias("avg_maintenance_cost_inr"))
    .orderBy("site_type", "severity").show())

+---------------+--------+------------+------------------------+
|      site_type|severity|ticket_count|avg_maintenance_cost_inr|
+---------------+--------+------------+------------------------+
|     Commercial|Critical|          37|                13424.73|
|     Commercial|    High|         130|                12293.07|
|     Commercial|     Low|         166|                12223.52|
|     Commercial|  Medium|         171|                 13706.6|
|     Industrial|Critical|          17|                 12296.2|
|     Industrial|    High|          63|                12838.87|
|     Industrial|     Low|          66|                12717.53|
|     Industrial|  Medium|         106|                 12043.9|
|Public Facility|Critical|          15|                 8986.84|
|Public Facility|    High|          48|                12905.28|
|Public Facility|     Low|          60|                11195.93|
|Public Facility|  Medium|          97|                13050.63|
|    Residential|Critical

12. Calculate average voltage_v and total outage_minutes by substation. Show the five substations with
the highest total outage minutes.

In [19]:
(readings_meter_master_df.groupBy("substation_id")
    .agg(F.round(F.avg("voltage_v"), 2).alias("avg_voltage_v"),
         F.sum("outage_minutes").alias("total_outage_minutes"))
    .orderBy(F.desc("total_outage_minutes")).limit(5).show())

+-------------+-------------+--------------------+
|substation_id|avg_voltage_v|total_outage_minutes|
+-------------+-------------+--------------------+
|        SUB10|       230.04|               91224|
|        SUB09|       230.05|               81734|
|        SUB04|       230.06|               75872|
|        SUB08|       229.88|               75581|
|        SUB03|       230.03|               61662|
+-------------+-------------+--------------------+



13. Calculate the number of Valid, Estimated and Rejected readings for each source_type, ordered by
reading count.

In [20]:
(readings_df.groupBy("source_type")
    .agg(F.sum((F.col("reading_status") == "Valid").cast("int")).alias("Valid"),
         F.sum((F.col("reading_status") == "Estimated").cast("int")).alias("Estimated"),
         F.sum((F.col("reading_status") == "Rejected").cast("int")).alias("Rejected"),
         F.count("*").alias("reading_count"))
    .orderBy(F.desc("reading_count")).show())

+-----------+-----+---------+--------+-------------+
|source_type|Valid|Estimated|Rejected|reading_count|
+-----------+-----+---------+--------+-------------+
| SmartMeter|65521|    10443|    3202|        79166|
|     Manual| 8965|     1405|     464|        10834|
+-----------+-----+---------+--------+-------------+



**SECTION 02 / ADVANCED DATAFRAME OPERATIONS**

**Tasks 14-25**

**D. Date and window functions**

14. Convert reading_timestamp into a Spark timestamp. Extract the year and month and calculate monthly
Valid energy consumption by region.

In [21]:
ts_df = (rms_df
    .withColumn("reading_ts", F.to_timestamp("reading_timestamp"))
    .withColumn("year", F.year("reading_ts"))
    .withColumn("month", F.month("reading_ts")))
(ts_df.filter(F.col("reading_status") == "Valid")
    .groupBy("region", "year", "month")
    .agg(F.round(F.sum("energy_kwh"), 2).alias("valid_kwh"))
    .orderBy("region", "year", "month").show())

+-------+----+-----+---------+
| region|year|month|valid_kwh|
+-------+----+-----+---------+
|Central|2026|    1| 31475.58|
|Central|2026|    2| 28161.41|
|Central|2026|    3| 29805.85|
|Central|2026|    4| 29475.34|
|Central|2026|    5| 30048.88|
|Central|2026|    6| 27536.85|
|  North|2026|    1| 48849.31|
|  North|2026|    2| 45444.25|
|  North|2026|    3| 51094.61|
|  North|2026|    4| 43451.02|
|  North|2026|    5| 48796.93|
|  North|2026|    6| 47072.33|
|  South|2026|    1| 97547.39|
|  South|2026|    2|  90864.8|
|  South|2026|    3|102258.77|
|  South|2026|    4| 97266.94|
|  South|2026|    5|101204.87|
|  South|2026|    6| 97699.97|
|   West|2026|    1| 59967.57|
|   West|2026|    2|  55807.9|
+-------+----+-----+---------+
only showing top 20 rows


15. Find the top three meters by total Valid energy within each substation using dense_rank().

In [22]:
top_3_meter = (readings_meter_master_df.filter(F.col("reading_status") == "Valid")
    .groupBy("substation_id", "meter_id").agg(F.sum("energy_kwh").alias("total_valid_kwh")))
w15 = Window.partitionBy("substation_id").orderBy(F.desc("total_valid_kwh"))
(top_3_meter.withColumn("rank", F.dense_rank().over(w15))
    .filter(F.col("rank") <= 3).orderBy("substation_id", "rank").show())

+-------------+--------+------------------+----+
|substation_id|meter_id|   total_valid_kwh|rank|
+-------------+--------+------------------+----+
|        SUB01|   M0130|           9364.98|   1|
|        SUB01|   M0110| 9259.750000000002|   2|
|        SUB01|   M0068| 8832.580000000005|   3|
|        SUB02|   M0071|          29242.04|   1|
|        SUB02|   M0145|28641.319999999992|   2|
|        SUB02|   M0133| 7060.140000000003|   3|
|        SUB03|   M0096| 9632.740000000003|   1|
|        SUB03|   M0080| 9525.940000000002|   2|
|        SUB03|   M0073| 9369.789999999995|   3|
|        SUB04|   M0038|          29505.88|   1|
|        SUB04|   M0099|29216.319999999996|   2|
|        SUB04|   M0098|28613.460000000014|   3|
|        SUB05|   M0094| 9659.639999999996|   1|
|        SUB05|   M0050| 9570.859999999997|   2|
|        SUB05|   M0008| 9524.780000000004|   3|
|        SUB06|   M0126|32573.380000000023|   1|
|        SUB06|   M0128|30748.840000000004|   2|
|        SUB06|   M0

16. Calculate cumulative Valid energy for each meter using a window ordered by reading timestamp and
reading ID.

In [23]:
w16 = (Window.partitionBy("meter_id").orderBy("reading_ts", "reading_id")
       .rowsBetween(Window.unboundedPreceding, Window.currentRow))
(ts_df.filter(F.col("reading_status") == "Valid")
    .withColumn("cumulative_kwh", F.sum("energy_kwh").over(w16))
    .select("meter_id", "reading_id", "reading_ts", "energy_kwh", "cumulative_kwh")
    .orderBy("meter_id", "reading_ts", "reading_id").show(20))

+--------+----------+-------------------+----------+------------------+
|meter_id|reading_id|         reading_ts|energy_kwh|    cumulative_kwh|
+--------+----------+-------------------+----------+------------------+
|   M0001| RD0083587|2026-01-01 01:15:00|     22.27|             22.27|
|   M0001| RD0086275|2026-01-01 03:30:00|      4.05|             26.32|
|   M0001| RD0088204|2026-01-01 09:45:00|     11.28|              37.6|
|   M0001| RD0005149|2026-01-02 04:45:00|     11.76|             49.36|
|   M0001| RD0035530|2026-01-02 08:15:00|     10.06|             59.42|
|   M0001| RD0070104|2026-01-02 18:30:00|     15.83|             75.25|
|   M0001| RD0006677|2026-01-03 00:30:00|     17.06|             92.31|
|   M0001| RD0021042|2026-01-03 06:00:00|     17.14|            109.45|
|   M0001| RD0032205|2026-01-03 08:30:00|     20.46|            129.91|
|   M0001| RD0074606|2026-01-03 18:15:00|      20.4|            150.31|
|   M0001| RD0019547|2026-01-04 19:30:00|      6.84|            

**E. Union and deduplication**

17. Combine both readings files using unionByName(). Compare row counts before and after the union.

In [24]:
combined_df = readings_df.unionByName(readings_extra_df)
print("Main:", readings_df.count(), "| Extra:", readings_extra_df.count(), "| Union:", combined_df.count())

Main: 90000 | Extra: 12000 | Union: 102000


18. Identify repeated reading_id values, remove duplicates using that key, and report the final unique
reading count.

In [25]:
dups = combined_df.groupBy("reading_id").count().filter("count > 1")
print("Repeated reading_id values:", dups.count())
dups.orderBy(F.desc("count")).show(5)

unique_df = combined_df.dropDuplicates(["reading_id"])
print("Final unique readings:", unique_df.count())

Repeated reading_id values: 700
+----------+-----+
|reading_id|count|
+----------+-----+
| RD0025848|    2|
| RD0045533|    2|
| RD0064921|    2|
| RD0056575|    2|
| RD0058815|    2|
+----------+-----+
only showing top 5 rows
Final unique readings: 101300


**F. Lazy execution, caching and persistence**

In [27]:
def timed(label, fn):
    t = time.time(); r = fn(); e = time.time() - t
    print(f"{label}: {e:.3f}s"); return r, e

19. Build a pipeline with filter(), withColumn(), groupBy() and orderBy(). Inspect it using explain() before
calling an action.

In [26]:
pipeline = (readings_df
    .filter(F.col("reading_status") == "Valid")
    .withColumn("kwh_x2", F.col("energy_kwh") * 2)
    .groupBy("meter_id").agg(F.sum("kwh_x2").alias("total"))
    .orderBy(F.desc("total")))
pipeline.explain()
pipeline.show(5) # action triggers execution

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total#1153 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total#1153 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=3055]
      +- HashAggregate(keys=[meter_id#18], functions=[sum(kwh_x2#1152)])
         +- Exchange hashpartitioning(meter_id#18, 200), ENSURE_REQUIREMENTS, [plan_id=3052]
            +- HashAggregate(keys=[meter_id#18], functions=[partial_sum(kwh_x2#1152)])
               +- Project [meter_id#18, (energy_kwh#20 * 2.0) AS kwh_x2#1152]
                  +- Filter (isnotnull(reading_status#24) AND (reading_status#24 = Valid))
                     +- FileScan csv [meter_id#18,energy_kwh#20,reading_status#24] Batched: false, DataFilters: [isnotnull(reading_status#24), (reading_status#24 = Valid)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(reading_status), EqualTo(reading_status,Valid)], ReadSchema: 

20. Cache Valid readings, run count() twice, and record the elapsed time for each action. Check is_cached.

In [28]:
valid_cached = readings_df.filter(F.col("reading_status") == "Valid").cache()
_, t1 = timed("count #1 (fills cache)", lambda: valid_cached.count())
_, t2 = timed("count #2 (from cache)", lambda: valid_cached.count())
print("is_cached:", valid_cached.is_cached)
valid_cached.unpersist()

count #1 (fills cache): 2.206s
count #2 (from cache): 0.130s
is_cached: True


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

21. Persist readings with outage_minutes > 0 using MEMORY_AND_DISK. Execute two actions and then
release the data using unpersist().

In [29]:
outage_df = readings_df.filter(F.col("outage_minutes") > 0).persist(StorageLevel.MEMORY_AND_DISK)
print("Rows:", outage_df.count())
print("Total outage minutes:", outage_df.agg(F.sum("outage_minutes")).first()[0])
print("Storage level:", outage_df.storageLevel)
outage_df.unpersist()
print("After unpersist, is_cached:", outage_df.is_cached)

Rows: 7653
Total outage minutes: 707706
Storage level: Disk Memory Serialized 1x Replicated
After unpersist, is_cached: False


22. Compare repeated execution of the same aggregation with and without caching. Record observations;
do not assume caching must be faster.

In [30]:
def agg_job(df):
    return df.groupBy("meter_id").agg(F.sum("energy_kwh").alias("t")).count()

base = readings_df.filter(F.col("reading_status") != "Rejected")
no_cache = [timed(f"no cache run {i+1}", lambda: agg_job(base))[1] for i in range(3)]

base_c = base.cache(); base_c.count()   # materialise
with_cache = [timed(f"cached run {i+1}", lambda: agg_job(base_c))[1] for i in range(3)]
base_c.unpersist()
print("Avg no-cache:", sum(no_cache)/3, "| Avg cached:", sum(with_cache)/3)

no cache run 1: 1.153s
no cache run 2: 0.715s
no cache run 3: 0.541s
cached run 1: 0.329s
cached run 2: 0.285s
cached run 3: 0.249s
Avg no-cache: 0.8027994632720947 | Avg cached: 0.2877475420633952


**G. Partitions and output**

23. Repartition the main readings DataFrame into eight partitions and show the number of rows in each
using spark_partition_id().

In [31]:
df8 = readings_df.repartition(8)
print("Partitions:", df8.rdd.getNumPartitions())
df8.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|11249|
|           1|11249|
|           2|11251|
|           3|11251|
|           4|11250|
|           5|11250|
|           6|11250|
|           7|11250|
+------------+-----+



24. Start from 12 partitions. Reduce to three using coalesce(3) and separately using repartition(3).
Compare the execution plans.

In [32]:
df12 = readings_df.repartition(12)
coalesce_3, repartition_3 = df12.coalesce(3), df12.repartition(3)
print("coalesce(3) partitions:", coalesce_3.rdd.getNumPartitions())
print("repartition(3) partitions:", repartition_3.rdd.getNumPartitions())
print("=== coalesce plan ==="); coalesce_3.explain()
print("=== repartition plan ==="); repartition_3.explain()

coalesce(3) partitions: 3
repartition(3) partitions: 3
=== coalesce plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=4150]
            +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:...
+- == Initial Plan ==
   Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=4150]
      +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v

25. Join cleaned readings with meter and substation details, then save the enriched DataFrame as Parquet
using four output partitions. Inspect the resulting part files.

In [33]:
enriched_df = (clean_df.join(meter_master, on="meter_id", how="inner")
                       .join(substation_df, on="substation_id", how="inner"))
out25 = "/content/output/enriched_readings"
enriched_df.repartition(4).write.mode("overwrite").parquet(out25)
print(sorted(os.listdir(out25)))
print("Part files:", len([f for f in os.listdir(out25) if f.startswith("part-")]))
spark.read.parquet(out25).count()

['._SUCCESS.crc', '.part-00000-f40d2bd8-c081-4863-9748-80913e231be7-c000.snappy.parquet.crc', '.part-00001-f40d2bd8-c081-4863-9748-80913e231be7-c000.snappy.parquet.crc', '.part-00002-f40d2bd8-c081-4863-9748-80913e231be7-c000.snappy.parquet.crc', '.part-00003-f40d2bd8-c081-4863-9748-80913e231be7-c000.snappy.parquet.crc', '_SUCCESS', 'part-00000-f40d2bd8-c081-4863-9748-80913e231be7-c000.snappy.parquet', 'part-00001-f40d2bd8-c081-4863-9748-80913e231be7-c000.snappy.parquet', 'part-00002-f40d2bd8-c081-4863-9748-80913e231be7-c000.snappy.parquet', 'part-00003-f40d2bd8-c081-4863-9748-80913e231be7-c000.snappy.parquet']
Part files: 4


86349

**SECTION 03 / INTEGRATED ASSESSMENT**

**Final Capstone**

**Energy Consumption & Reliability Report**

Build an end-to-end PySpark pipeline that produces a monthly report for each region and site_type. Use
the main and additional readings files and the relevant master datasets.

01. Combine the two readings datasets and deduplicate by reading_id.

02. Apply a documented cleaning rule for null energy values. Exclude Rejected readings from billable
consumption.

03. Join meter, substation and tariff information using the appropriate keys.

04. Create a month column from reading_timestamp and calculate total kWh and estimated energy
charges.

05. Calculate total outage minutes and the percentage of readings that report an outage for each region
and site type.

06. Rank site types within each region by total billable energy charges.

07. Cache or persist the final aggregated report, run two actions, and release the cached data.

08. Write the report as Parquet with two output partitions.

In [34]:
# 1. Combine + dedupe
cap = readings_df.unionByName(readings_extra_df).dropDuplicates(["reading_id"])

# 2. Cleaning rule for null energy values
cap = cap.dropna(subset=["energy_kwh"])

# 3. Join meter, substation and tariff
cap = (cap.join(meter_dim, on="meter_id", how="inner")
          .join(sub_dim, on="substation_id", how="inner")
          .join(tariff_dim, on="tariff_id", how="inner"))

# 4. Month + billing
cap = (cap.withColumn("month", F.date_format(F.to_timestamp("reading_timestamp"), "yyyy-MM"))
          .withColumn("is_billable", F.col("reading_status") != "Rejected")
          .withColumn("billable_kwh", F.when(F.col("is_billable"), F.col("energy_kwh")))
          .withColumn("energy_charge_inr", F.col("billable_kwh") * F.col("rate_per_kwh_inr"))
          .withColumn("has_outage", (F.col("outage_minutes") > 0).cast("int")))

# 5. Aggregation
report = (cap.groupBy("region", "site_type", "month")
    .agg(F.round(F.sum("billable_kwh"), 2).alias("total_billable_kwh"),
         F.round(F.sum("energy_charge_inr"), 2).alias("total_energy_charge_inr"),
         F.sum("outage_minutes").alias("total_outage_minutes"),
         F.count("*").alias("reading_count"),
         F.round(100 * F.avg("has_outage"), 2).alias("outage_reading_pct")))

# 6. Rank site types within each region by total billable charges (all months)
rank_df = (report.groupBy("region", "site_type")
    .agg(F.sum("total_energy_charge_inr").alias("region_site_charge"))
    .withColumn("site_type_rank",
        F.dense_rank().over(Window.partitionBy("region").orderBy(F.desc("region_site_charge")))))
final_report = (report.join(rank_df.select("region", "site_type", "site_type_rank"), ["region", "site_type"])
                      .orderBy("region", "site_type_rank", "month"))

# 7. Persist, two actions, release
final_report.persist(StorageLevel.MEMORY_AND_DISK)
print("Report rows:", final_report.count())
final_report.show()
rank_df.orderBy("region", "site_type_rank").show()
final_report.unpersist()

# 8. Parquet with two output partitions
out_cap = "/content/output/energy_report"
final_report.coalesce(2).write.mode("overwrite").parquet(out_cap)
print(sorted(os.listdir(out_cap)))

Report rows: 120
+-------+-----------+-------+------------------+-----------------------+--------------------+-------------+------------------+--------------+
| region|  site_type|  month|total_billable_kwh|total_energy_charge_inr|total_outage_minutes|reading_count|outage_reading_pct|site_type_rank|
+-------+-----------+-------+------------------+-----------------------+--------------------+-------------+------------------+--------------+
|Central| Industrial|2026-01|          24871.22|               226328.1|                4396|          432|             11.34|             1|
|Central| Industrial|2026-02|          21460.69|              195292.28|                3763|          375|             10.67|             1|
|Central| Industrial|2026-03|          22963.22|               208965.3|                3438|          396|              10.1|             1|
|Central| Industrial|2026-04|          22201.23|              202031.19|                3058|          372|               7.8|     